**HuggingFace Pipelines:**
- convenient api to download and use pretrained pipelines for various tasks. each pipeline contains a pretrained model along with its corresponding preprocessing and postprocessing modules. for example - create a sentiment analysis pipeline and run it on the first 10IMDB reviews in the traaining set..

In [1]:
from transformers import pipeline
from datasets import load_dataset

In [14]:
import torch
import torch.nn as nn

In [2]:
data = load_dataset("imdb")
split = data['train'].train_test_split(train_size=0.8, seed=42)
imdb_train_set, imdb_val_set = split['train'], split['test']
imdb_test_set = data['test']

In [6]:
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
classifier_imdb = pipeline("sentiment-analysis", model=model_name, truncation=True, max_length=512)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Device set to use mps:0


In [7]:
classifier_imdb(imdb_test_set['text'][:20])

[{'label': 'NEGATIVE', 'score': 0.999616265296936},
 {'label': 'NEGATIVE', 'score': 0.6170646548271179},
 {'label': 'NEGATIVE', 'score': 0.9997100234031677},
 {'label': 'NEGATIVE', 'score': 0.995756208896637},
 {'label': 'POSITIVE', 'score': 0.996307373046875},
 {'label': 'NEGATIVE', 'score': 0.9966711401939392},
 {'label': 'NEGATIVE', 'score': 0.958416759967804},
 {'label': 'NEGATIVE', 'score': 0.9994093179702759},
 {'label': 'NEGATIVE', 'score': 0.9996923208236694},
 {'label': 'NEGATIVE', 'score': 0.99977046251297},
 {'label': 'NEGATIVE', 'score': 0.9997914433479309},
 {'label': 'NEGATIVE', 'score': 0.9940081834793091},
 {'label': 'NEGATIVE', 'score': 0.9997391104698181},
 {'label': 'NEGATIVE', 'score': 0.9996050000190735},
 {'label': 'NEGATIVE', 'score': 0.9997557997703552},
 {'label': 'NEGATIVE', 'score': 0.9913800358772278},
 {'label': 'NEGATIVE', 'score': 0.9960663914680481},
 {'label': 'NEGATIVE', 'score': 0.9991810917854309},
 {'label': 'POSITIVE', 'score': 0.6568294763565063},

evaluate:

In [8]:
o = classifier_imdb(imdb_test_set['text'][:20])

In [10]:
o2 = list(map(lambda x: 0 if x['label']=='NEGATIVE' else 1, o))

In [18]:
o3 = torch.tensor(o2)

In [19]:
lab = torch.tensor(imdb_test_set['label'][:20])

In [21]:
o3.shape

torch.Size([20])

In [24]:
(o3==lab).sum().item()/20

0.9

Bias and Fairness

In [37]:
classifier_imdb(["I am from Islamabad"])

[{'label': 'POSITIVE', 'score': 0.9791147708892822}]

many text classification tasks other than sentiment analysis. For example a model fine-tuned on the multi-genre language inference dataset can classify a pair of texts (each ending with a separation token [SEP]) into 3 classes

In [38]:
model_name = "huggingface/distilbert-base-uncased-finetuned-mnli"
classifier_mnli = pipeline("text-classification", model=model_name)

config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use mps:0


In [39]:
classifier_mnli([
    "She loves me. [SEP] She loves me not. [SEP]",
    "Alice just woke up. [SEP] Alice is awake. [SEP]",
    "I like dogs. [SEP] Everyone likes dogs. [SEP]"
])

[{'label': 'contradiction', 'score': 0.9717152714729309},
 {'label': 'entailment', 'score': 0.9119169116020203},
 {'label': 'neutral', 'score': 0.9509280323982239}]